# Capstone Research Paper — Search Traffic Decay & Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaineshchaurasiya20/FlyRank_Ml_Assignment/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

- **Author:** Jainesh Chaurasiya (<aw888117@gmail.com>)
- **Track:** ML Engineering Intern, FlyRank AI
- **Lane:** Content Refresh & Search Decay Prioritization
- **Repository:** [FlyRank_Ml_Assignment](https://github.com/jaineshchaurasiya20/FlyRank_Ml_Assignment)
- **Deployed Paper:** [https://jaineshchaurasiya20.github.io/FlyRank_Ml_Assignment/](https://jaineshchaurasiya20.github.io/FlyRank_Ml_Assignment/)

---

## 0. Abstract

Modern search engines continuously recalibrate organic rankings, causing mature website content to experience silent traffic decay that editorial teams typically detect months too late. Working with the FlyRank AI Search Intelligence dataset spanning 79M+ warehouse event rows and a 30,000-page cross-client benchmark across 32 enterprise domains, we evaluate whether machine learning can prioritize recoverable content decay more effectively than heuristic rules. We trained and audited a Gradient Boosted Decision Tree (GBDT) pipeline under a strict client-holdout validation protocol, explicitly eliminating label leakage and domain-identity memorization. On unseen client domains with a 39.1% decay base rate, the model achieves a Precision@50 of 86.0% and a PR-AUC of 0.6772, significantly outperforming the production heuristic rule baseline (Precision@50 = 22.0%, PR-AUC = 0.4700). We operationalize these predictive rankings into a human-in-the-loop Content Action Playbook that maps multi-dimensional search signals into concrete editorial workflows with clear cost-benefit economics, safety guardrails, and retrain tripwires.

## 1. Question & Problem Framing

*The research question and the decision it supports.*

In [1]:
# Section 1: Problem framing verification & business scale
print("=== RESEARCH QUESTION & DECISION FRAMING ===")
print("Primary Research Question:")
print("Can machine learning models operating on backward-looking search metrics identify decaying organic URLs ")
print("across unseen client domains more accurately than hand-crafted heuristic rules, and how can these scores ")
print("be translated into high-ROI human editorial workflows?\n")

print("Core Decision Supported:")
print("- Unit of Analysis: One published URL (content item) aggregated over trailing performance windows.")
print("- Operational Decision: Out of thousands of aging library pages, which 50 should editors update first this sprint?")
print("- Cost of Error:")
print("    * False Positive: ~1.5 to 4.0 wasted editorial hours updating stable or structurally broken URLs.")
print("    * False Negative: Continued erosion of valuable organic traffic to competing search results.")

=== RESEARCH QUESTION & DECISION FRAMING ===
Primary Research Question:
Can machine learning models operating on backward-looking search metrics identify decaying organic URLs 
across unseen client domains more accurately than hand-crafted heuristic rules, and how can these scores 
be translated into high-ROI human editorial workflows?

Core Decision Supported:
- Unit of Analysis: One published URL (content item) aggregated over trailing performance windows.
- Operational Decision: Out of thousands of aging library pages, which 50 should editors update first this sprint?
- Cost of Error:
    * False Positive: ~1.5 to 4.0 wasted editorial hours updating stable or structurally broken URLs.
    * False Negative: Continued erosion of valuable organic traffic to competing search results.


## 2. Data Contract & Provenance

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate verified data artifact
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("FlyRank_Ml_Assignment/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {len(df):,} rows x {df.shape[1]} columns across {df['client_id'].nunique()} client domains.")

# Data Contract Verification
print("\n--- DATA PROVENANCE & SUMMARY TABLE ---")
data_summary = pd.DataFrame([
    {'Dimension': 'Upstream Warehouse Grain', 'Specification': 'report_date x client_id x content_id (78.8M+ daily performance records)'},
    {'Dimension': 'Modeling Slice Grain', 'Specification': 'One row per pseudonymized content item (URL)'},
    {'Dimension': 'Dataset Scope', 'Specification': '30,000 rows across 32 pseudonymized client domains'},
    {'Dimension': 'Time Horizon', 'Specification': 'Trailing 90-day activity (days 1-90) partitioned into 30d recent vs 30d prior'},
    {'Dimension': 'Label Definition', 'Specification': 'is_declining_label = (trend_direction == "down") [traffic drop > 20%]'},
    {'Dimension': 'Excluded Columns (Leakage)', 'Specification': 'trend_direction, trend_pct (label sources), content_id, client_id (IDs)'},
    {'Dimension': 'Excluded Columns (Operational)', 'Specification': 'provider_used, model_used, ga4_data_available, health_score (product flags)'}
])
display(data_summary)


Loaded dataset: 30,000 rows x 44 columns across 32 client domains.

--- DATA PROVENANCE & SUMMARY TABLE ---


,Dimension,Specification
0,Upstream Warehouse Grain,report_date x client_id x content_id (78.8M+ daily performance records)
1,Modeling Slice Grain,One row per pseudonymized content item (URL)
2,Dataset Scope,"30,000 rows across 32 pseudonymized client domains"
3,Time Horizon,Trailing 90-day activity (days 1-90) partitioned into 30d recent vs 30d prior
4,Label Definition,"is_declining_label = (trend_direction == ""down"") [traffic drop > 20%]"
5,Excluded Columns (Leakage),"trend_direction, trend_pct (label sources), content_id, client_id (IDs)"
6,Excluded Columns (Operational),"provider_used, model_used, ga4_data_available, health_score (product flags)"


## 3. Methodology & Validation Protocol

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
# Section 3: Feature Engineering, Client-Holdout Split, and Leakage Audit Harness
print("=== METHODOLOGY & VALIDATION SETUP ===")

# 1. Feature Engineering with Missingness & Position Cleanups
df['position_clean'] = np.where(df['avg_position'] == 0, 50.0, df['avg_position'])
df['decay_ratio_lag'] = (df['impressions_last_30d'] + 1) / (df['impressions_prev_30d'] + 1)
df['bounce_proxy'] = 100.0 - df['engagement_rate']

FEATURE_COLS = [
    'clicks_last_30d',
    'impressions_last_30d',
    'decay_ratio_lag',
    'position_clean',
    'ctr',
    'sessions_last_30d',
    'bounce_proxy'
]
X = df[FEATURE_COLS].fillna(0)
y = (df['trend_direction'] == 'down').astype(int)

# 2. Client-Holdout Split (Honest Generalization to Unseen Domains)
VAL_CLIENTS = ['client_0b918943df', 'client_1a6562590e', 'client_98a3ab7c34', 'client_f74efabef1', 'client_d4735e3a26', 'client_4fc82b26ae']
train_idx = ~df['client_id'].isin(VAL_CLIENTS)
test_idx = df['client_id'].isin(VAL_CLIENTS)

print(f"Training set: {train_idx.sum():,} rows across {df[train_idx]['client_id'].nunique()} clients (Base rate = {y[train_idx].mean():.1%})")
print(f"Holdout test set: {test_idx.sum():,} rows across {len(VAL_CLIENTS)} clients (Base rate = {y[test_idx].mean():.1%})")

# 3. Leakage Guard Confirmation
assert 'trend_pct' not in FEATURE_COLS, "CRITICAL: trend_pct found in features!"
assert 'trend_direction' not in FEATURE_COLS, "CRITICAL: trend_direction found in features!"
print("Feature leakage guard: PASSED. Strictly backward-looking signals used.")


=== METHODOLOGY & VALIDATION SETUP ===
Training set: 27,675 rows across 26 clients (Base rate = 55.5%)
Holdout test set: 2,325 rows across 6 clients (Base rate = 39.1%)
Feature leakage guard: PASSED. Strictly backward-looking signals used.


## 4. Results (Model vs Baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
import json

# Load precomputed, validated benchmark metrics from work/outputs/
METRICS_PATH = Path("work/outputs/model_comparison_metrics.json")
if not METRICS_PATH.exists():
    METRICS_PATH = Path("../outputs/model_comparison_metrics.json")

with open(METRICS_PATH, "r") as f:
    comp_data = json.load(f)

results_table = []
for model_name, m in comp_data["models"].items():
    results_table.append({
        'Model Architecture': model_name,
        'PR-AUC (Holdout)': round(m['PR-AUC'], 4),
        'ROC-AUC (Holdout)': round(m['ROC-AUC'], 4),
        'Precision@20': f"{m['Precision@20']:.1%}",
        'Precision@50': f"{m['Precision@50']:.1%}",
        'Precision@100': f"{m['Precision@100']:.1%}",
        'Accuracy': f"{m['Accuracy']:.1%}"
    })

results_df = pd.DataFrame(results_table)
print(f"Holdout Test Size: {comp_data['test_rows']:,} rows | Unseen Base Rate: {comp_data['test_base_rate']:.1%}")
display(results_df)

print("\nKey Findings Observed:")
print("1. Baseline Rule vs GBDT: Precision@50 rises from 22.0% -> 86.0% (3.9x lift over the heuristic baseline).")
print("2. Discrimination: Holdout PR-AUC improves from 0.4700 -> 0.6772 (+0.2072 absolute gain over rule).")
print("3. Generalization Audit: Moving from random split to client-holdout lowers PR-AUC by 0.1368, quantifying domain memorization.")


Holdout Test Size: 2,325 rows | Unseen Base Rate: 39.1%


,Model Architecture,PR-AUC (Holdout),ROC-AUC (Holdout),Precision@20,Precision@50,Precision@100,Accuracy
0,Baseline Hand Rule (W4),0.4700,0.6262,25.0%,22.0%,29.0%,61.1%
1,Logistic Regression,0.5280,0.7005,45.0%,34.0%,45.0%,67.4%
2,Decision Tree (depth=5),0.5753,0.7415,80.0%,68.0%,65.0%,67.7%
3,Random Forest,0.6453,0.7577,85.0%,84.0%,80.0%,66.7%
4,Gradient Boosting,0.6772,0.7753,95.0%,86.0%,85.0%,67.4%


## 5. Limitations & Honest Claim Framing

*What this work cannot claim.*

In [5]:
# Section 5: Documenting Empirical Boundaries & Limitations
print("=== EMPIRICAL BOUNDARIES & CLAIM CONSTRAINTS ===")
limitations = pd.DataFrame([
    {'Constraint Area': 'No Causal Guarantees', 'Measured Boundary': 'Observational only; models predict correlation with historical decay, not guaranteed rank recovery post-refresh.'},
    {'Constraint Area': 'Low-Volume Inefficacy', 'Measured Boundary': 'Noise dominates at <20 impressions/30d; 29.6% of portfolio is truncated from automated recommendations.'},
    {'Constraint Area': 'Catalog Bias', 'Measured Boundary': 'Trained on 32 B2B and content-publishing sites; uncalibrated for high-velocity seasonal retail or event catalogs.'},
    {'Constraint Area': 'Cold-Start Blindness', 'Measured Boundary': 'Content items <90 days old are omitted; initial Google indexing volatility cannot be reliably ranked.'},
    {'Constraint Area': 'Unmodeled Exogenous Shocks', 'Measured Boundary': 'Google core updates, brand PR spikes, and UI carousel changes require manual editorial awareness.'}
])
display(limitations)


=== EMPIRICAL BOUNDARIES & CLAIM CONSTRAINTS ===


,Constraint Area,Measured Boundary
0,No Causal Guarantees,"Observational only; models predict correlation with historical decay, not guaranteed rank recovery post-refresh."
1,Low-Volume Inefficacy,Noise dominates at <20 impressions/30d; 29.6% of portfolio is truncated from automated recommendations.
2,Catalog Bias,Trained on 32 B2B and content-publishing sites; uncalibrated for high-velocity seasonal retail or event catalogs.
3,Cold-Start Blindness,Content items <90 days old are omitted; initial Google indexing volatility cannot be reliably ranked.
4,Unmodeled Exogenous Shocks,"Google core updates, brand PR spikes, and UI carousel changes require manual editorial awareness."


## 6. Ranked Recommendations (Content Action Playbook)

*The action playbook output — the paper's recommendations section.*

In [6]:
# Section 6: Action Playbook Archetype Breakdown
PLAYBOOK_PATH = Path("work/outputs/action_playbook_summary.json")
if not PLAYBOOK_PATH.exists():
    PLAYBOOK_PATH = Path("../outputs/action_playbook_summary.json")

with open(PLAYBOOK_PATH, "r") as f:
    pb_data = json.load(f)

actions_df = pd.DataFrame([
    {'Recommended Action': act, 'Count': count, 'Portfolio Share': f"{count / pb_data['portfolio_size'] * 100:.1f}%"}
    for act, count in pb_data['action_distribution'].items()
]).sort_values(by='Count', ascending=False)

print(f"Portfolio Size: {pb_data['portfolio_size']:,} URLs | Actionable Candidates: {pb_data['actionable_queue_size']:,} URLs")
print(f"Estimated Top-100 Editorial Review Labor: {pb_data['top_100_estimated_review_hours']:.1f} hours")
display(actions_df)


Portfolio Size: 30,000 URLs | Actionable Candidates: 21,120 URLs
Estimated Top-100 Editorial Review Labor: 95.0 hours


,Recommended Action,Count,Portfolio Share
0,DO_NOT_AUTOMATE,8880,29.6%
1,TITLE_SNIPPET_OPTIMIZATION,5997,20.0%
2,REFRESH_AUTHORITY_EXPANSION,3322,11.1%
3,SCHEDULED_REVIEW,3238,10.8%
4,TECHNICAL_AND_CONTENT_AUDIT,2958,9.9%
5,MONITOR_AND_PROTECT,2948,9.8%
6,COMPREHENSIVE_CONTENT_REFRESH,2207,7.4%
7,MANDATORY_HUMAN_AUDIT,450,1.5%


## 7. Artifacts the Paper Embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
from IPython.display import Image, display as nb_display

# Verify and display publication figures
figs = [
    ("Figure 1: Model vs Baseline Discrimination", Path("work/figures/model_vs_baseline_comparison.png")),
    ("Figure 2: Validation Split Generalization Gap", Path("work/figures/honest_validation_split_audit.png")),
    ("Figure 3: Portfolio Action Distribution", Path("work/figures/playbook_action_distribution.png")),
    ("Figure 4: Traffic Opportunity vs Decay Risk Matrix", Path("work/figures/playbook_priority_matrix.png"))
]

for title, p in figs:
    if p.exists():
        print(f"{title} (Verified at: {p})")
    else:
        print(f"{title} (MISSING at: {p})")


Figure 1: Model vs Baseline Discrimination (Verified at: work\figures\model_vs_baseline_comparison.png)
Figure 2: Validation Split Generalization Gap (Verified at: work\figures\honest_validation_split_audit.png)
Figure 3: Portfolio Action Distribution (Verified at: work\figures\playbook_action_distribution.png)
Figure 4: Traffic Opportunity vs Decay Risk Matrix (Verified at: work\figures\playbook_priority_matrix.png)


## 8. ML-12 Capstone Presentation & Communication Cuts

### A. 5-Minute Technical Demo Outline
- **0:00 - 1:00 (The Problem):** Why organic search decay is invisible until clicks crash; why fixed heuristic rules misallocate scarce copywriter hours.
- **1:00 - 2:00 (The Data & Leakage Trap):** Demonstrating the 79M warehouse pipeline and how naive split / label construction leaks without client-holdout partitioning.
- **2:00 - 3:15 (Model vs Baseline):** Showing the 3.9x Precision@50 gain (22.0% -> 86.0%) and the honest generalization gap (-0.1368 PR-AUC drop on holdout clients).
- **3:15 - 4:15 (The Content Action Playbook):** Live walkthrough of the Priority Queue: Title/Snippet optimizations vs Striking-distance refreshes vs No-Go automation stops.
- **4:15 - 5:00 (Limits & Tripwires):** What we cannot claim (no causal guarantees, volume floors) and the monitoring tripwires (PSI > 0.25).

---

### B. Social-Post Cut (LinkedIn / X Summary)
> **Can ML predict organic search decay before traffic collapses?**
>
> Over 90 days, we analyzed 30,000 pages across 32 enterprise domains using FlyRank's 79M+ search console warehouse. The finding: standard heuristic rules fail on unseen sites (Precision@50 = 22.0%).
>
> By training a Gradient Boosted model evaluated under a strict client-holdout split, we achieved an observed Precision@50 of 86.0% and PR-AUC of 0.6772—nearly 4x higher triage accuracy.
>
> More importantly: raw predictions don't edit articles. We translated the model into a Content Action Playbook that maps URLs into concrete workflows (Title optimizations, striking-distance expansions, and a strict no-go automation list).
>
> Full interactive paper & reproducible code: https://jaineshchaurasiya20.github.io/FlyRank_Ml_Assignment/

---

### C. Employer-Facing 3-Sentence Summary
I engineered an end-to-end ML decay prioritization pipeline and Content Action Playbook trained on FlyRank's 79M-row search performance dataset across 32 client domains. By implementing an honest client-holdout validation harness that eliminated target leakage and domain-identity memorization, my Gradient Boosted model delivered an observed Precision@50 of 86.0% on unseen domains (a 3.9x improvement over the heuristic rule baseline). I translated these predictions into a production-ready decision-support system featuring automated reason codes, cost-benefit labor estimations, and drift tripwires.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.